In [1]:
from stark_qa import load_skb

import random
import csv

import random
from ollama import generate

from neo4j import GraphDatabase

OLLAMA_LLM = "gemma4:26b"
OUTPUT_FILE = "../qa_datasets/text_amazon.csv"

DATASET_NAME = "amazon"
skb = load_skb(DATASET_NAME, download_processed=True)

/home/wagnerr/.venvstark/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading from /home/wagnerr/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/skb/amazon/processed!
Loading cached graph with meta link types ['brand', 'category', 'color']


In [ ]:
from typing import Literal

from pydantic import BaseModel


class QueryAmbiguityCheck(BaseModel):
    decision: Literal["KEEP", "DISCARD"]


NUM_QUERIES = 10
QUERY_GEN_PROMPT = """You are an intelligent assistant that generates queries about Amazon items.
I will provide you with a textual document associated with a product entity in a Amazon product recommendation knowledge graph.
Your task is to create a natural-sounding customer query that leads to this entity as the answer.
The graph contains a lot of similar products, so make sure your query is specific enough to only lead to this product as the only answer. 
Avoid using the product name in your query.

Document:
{doc}

Query: """
NEO4J_URI = "bolt://localhost:17687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "X"
SIMILAR_NODES_CYPHER = """MATCH (p1:product {{id: '{id}'}})
MATCH (p2:product)
  SEARCH p2 IN (
    VECTOR INDEX documents
    FOR p1.embedding
    LIMIT 5
  )
RETURN p2.document AS document"""

QUERY_AMBIGUITY_PROMPT = """You are an intelligent assistant that checks if a query is ambiguous.
I will provide you with a list of documents, each describing a different product.
Document #0 is the document the query was generated with.
The other documents are semantically similar documents.

Please decide:
- "KEEP" if the query is specific enough to only lead to document #0 as the answer.
- "DISCARD" if the query could be understood to lead to one of the other documents as well. 

Query: {query}

Documents:
{documents}
"""


driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

product_ids = skb.get_node_ids_by_type("product")
random_product_ids = random.sample(product_ids, k=NUM_QUERIES)

with open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=["id", "query", "answer_ids"])
    writer.writeheader()

    for i, pid in enumerate(random_product_ids):
        query_gen_prompt = QUERY_GEN_PROMPT.format(
            doc=skb.get_doc_info(pid, add_rel=False)
        )
        response = generate(
            model=OLLAMA_LLM,
            prompt=query_gen_prompt,
            options={"temperature": 0.0, "seed": 7},
            think="low",
        )
        generated_query = response.response
        cypher_query = SIMILAR_NODES_CYPHER.format(id=pid)
        records, _, _ = driver.execute_query(
            cypher_query,
        )
        similar_docs_string = ""
        for j, r in enumerate(records):
            similar_docs_string += f"-----Doc #{j}-----\n"
            similar_docs_string += r["document"] + "\n"

        query_ambiguity_prompt = QUERY_AMBIGUITY_PROMPT.format(
            query=generated_query, documents=similar_docs_string
        )
        print(query_ambiguity_prompt)
        response = generate(
            model=OLLAMA_LLM,
            prompt=query_ambiguity_prompt,
            options={"temperature": 0.0, "seed": 7},
            think="low",
            format=QueryAmbiguityCheck.model_json_schema(),
        )
        ambiguity_check = QueryAmbiguityCheck.model_validate_json(response.response)
        print(ambiguity_check)
        if ambiguity_check.decision == "KEEP":
            writer.writerow(
                {
                    "id": pid,
                    "query": generated_query,
                    "answer_ids": [pid],
                }
            )

You are an intelligent assistant that checks if a query is ambiguous.
I will provide you with a list of documents, each describing a different product.
Document #0 is the document the query was generated with.
The other documents are semantically similar documents.

Please decide:
- "KEEP" if the query is specific enough to only lead to document #0 as the answer.
- "DISCARD" if the query could be understood to lead to one of the other documents as well. 

Query: I am looking for Nordica alpine ski boots for kids in size US 1 (mondo 20.5) with a flex rating of 35, suitable for intermediate to advanced intermediate skiers.

Documents:
-----Doc #0-----
- product: kids ski boots US 1 mondo 20.5 NEW NORDICA GPT3 new GPT 3
- brand: Nordica
- description: kids ski boots US 1 mondo 20.5 NEW NORDICA GPT3 Product Description Nordica GPT3 Kid's Ski Boots - The Nordica GPT3 Alpine Ski Boot for Youth is a scaled-down, four-buckle, front-entry boot designed for kids with larger feet. Shorter cuff le